# S'mile — past price patterns by time of day

What travelers actually paid at each hour, from the same public data the price models learned from. The app draws this next to each model's prediction so you can see the past pattern and the predicted one side by side.

- **Rideshare:** median fare per mile by request hour, for weekdays and weekends, in trip-distance bands (NYC TLC high-volume trips, same two months and sample as the model). The app scales it by the trip's distance.
- **Flights:** median price of nonstop departures at each hour relative to the same route's median that day (Expedia fares, New Zealand, CC BY 4.0), rescaled so the day averages 1.0. The app multiplies it by the route-level fare.

Run after `smile_price_models.ipynb` (it reuses that notebook's downloaded data). Output: `models/price_history.json`.

In [ ]:
import os, sys, json
sys.modules["numexpr"] = None; sys.modules["bottleneck"] = None
import numpy as np, pandas as pd
ROOT = os.path.expanduser("~/smile-price-models"); DATA = f"{ROOT}/data"; OUT = f"{ROOT}/models"
SEED = 7
MONTHS = ["2026-02", "2026-08"]
BANDS = [0, 2, 5, 10, 20, 40, 1000]      # trip miles
MIN_TRIPS = 30

In [ ]:
cols = ["hvfhs_license_num", "request_datetime", "trip_miles", "trip_time", "base_passenger_fare"]
frames = []
for ym in MONTHS:
    df = pd.read_parquet(f"{DATA}/fhvhv_{ym}.parquet", columns=cols)
    df = df[df.hvfhs_license_num.isin(["HV0003", "HV0005"])]
    frames.append(df.sample(n=min(len(df), 3_000_000), random_state=SEED))
raw = pd.concat(frames, ignore_index=True)
minutes = raw.trip_time / 60
ok = raw.trip_miles.between(0.3, 100) & minutes.between(2, 240) & raw.base_passenger_fare.between(2.5, 500) & ((raw.base_passenger_fare / raw.trip_miles) < 30)
r = raw[ok]
t = pd.DataFrame({"hour": r.request_datetime.dt.hour, "weekend": (r.request_datetime.dt.dayofweek >= 5).astype(int),
                  "band": pd.cut(r.trip_miles, BANDS, right=False, labels=False), "fpm": r.base_passenger_fare / r.trip_miles})
g = t.groupby(["band", "weekend", "hour"]).fpm.agg(["median", "size"]).reset_index()
rideshare = {"bands_miles": BANDS, "min_trips": MIN_TRIPS, "rows": int(len(t)), "fare_per_mile": {}}
for b in range(len(BANDS) - 1):
    for w, name in ((0, "weekday"), (1, "weekend")):
        s = g[(g.band == b) & (g.weekend == w)].set_index("hour")
        rideshare["fare_per_mile"][f"{b}:{name}"] = [round(float(s.loc[h, "median"]), 4) if h in s.index and s.loc[h, "size"] >= MIN_TRIPS else None for h in range(24)]
print("rows:", len(t))
print(pd.DataFrame({k: v for k, v in rideshare["fare_per_mile"].items() if k.startswith("2:")}).round(2).to_string())

In [ ]:
nz = pd.read_csv(f"{DATA}/nz_airfares.csv", encoding="utf-8-sig")
nz.columns = ["date", "dep", "dep_time", "arr", "arr_time", "duration", "direct", "transit", "baggage", "airline", "fare"]
nz = nz.dropna(subset=["date", "dep", "arr", "dep_time", "fare"]).copy()
nz["date"] = pd.to_datetime(nz.date, format="%d/%m/%Y")
nz["hour"] = pd.to_datetime(nz.dep_time, format="%I:%M %p").dt.hour
nz = nz[nz.fare.between(30, 3000)]
nz["rel"] = nz.fare / nz.groupby([nz.dep + "-" + nz.arr, "date"]).fare.transform("median")
nz["weekend"] = (nz.date.dt.dayofweek >= 5).astype(int)
ns = nz[nz.direct == "(Direct)"]
flights = {"hours": list(range(5, 23)), "rows": int(len(ns)), "min_flights": MIN_TRIPS, "factor": {}}
for w, name in ((0, "weekday"), (1, "weekend")):
    s = ns[ns.weekend == w].groupby("hour").rel.agg(["median", "size"])
    vals = [float(s.loc[h, "median"]) if h in s.index and s.loc[h, "size"] >= MIN_TRIPS else None for h in flights["hours"]]
    known = [v for v in vals if v is not None]
    mean = sum(known) / len(known)
    flights["factor"][name] = [round(v / mean, 4) if v is not None else None for v in vals]
print("nonstop fares:", len(ns)); print(pd.DataFrame(flights["factor"], index=flights["hours"]).to_string())

In [ ]:
history = {"version": pd.Timestamp.now().strftime("%Y-%m-%d"), "rideshare": rideshare, "flights": flights}
with open(f"{OUT}/price_history.json", "w") as f:
    json.dump(history, f, separators=(",", ":"))
print("wrote", f"{OUT}/price_history.json", os.path.getsize(f"{OUT}/price_history.json"), "bytes")